# Framing and Feasibility Audit: NYC Mobility Friction

**Portfolio Project**  
**Author:** Vinicius Miozzo

**Date:** April 2026

---

## Objective

This project uses **NYC Yellow Taxi trip records** to identify taxi zones that show signs of **operational strain or inefficient service patterns**.

The goal is **not** to explain all mobility issues in New York City.  
The goal is to answer a practical prioritization question:

**Which taxi zones should be prioritized for deeper operational review?**

---

## Decision Context

City agencies and transportation operators cannot review every taxi zone at once.

This project acts as a **prioritization tool** that flags zones where taxi activity suggests possible operational friction, such as:

- Unusually long trip times relative to distance  
- Persistent pickup/dropoff imbalance  
- Low pickup activity relative to surrounding demand patterns  
- Concentration of late-night or irregular service patterns  
- Repeated signs of weak zone-level performance over time  

These zones may be candidates for:
- Curb management review
- Pickup/dropoff policy review
- Taxi stand evaluation
- Late-night service monitoring
- Deeper operational investigation

---

## Working Question

**Which taxi zones should be prioritized for deeper operational review?**

---

## Current Scope

- **Taxi data**: January 2025 (prototype month)  
- Uses **one month of taxi trip data** to validate the workflow, define useful indicators, and test whether taxi zones can support a stable zone-level prioritization framework.  
- At this stage, the project is a **prototype**, not a final policy recommendation.  
- A longer time window (multiple months) will be needed before making stable conclusions about persistent zone-level patterns.

---

## Data Source

- **NYC Taxi Trip Records** (Yellow Taxi)  
  Used to measure trip activity, trip outcomes, and zone-level operational patterns.

---

## Analytical Approach

The project builds a **zone-level view** of taxi operations using indicators such as:

- Pickup volume  
- Dropoff volume  
- Pickup/dropoff imbalance  
- Trip duration  
- Trip distance  
- Duration per mile  
- Late-night trip share  
- Invalid or extreme-trip rate  
- Day-level persistence of high-friction conditions  

---

## Proposed Unit of Analysis

**Taxi Zone × Day**

This unit is:
- More informative than monthly aggregates  
- Less noisy than trip-level analysis  
- Easy to roll up into zone-week or zone-month views later  

---

## Candidate Metrics (Taxi-side)

Per taxi zone per day:
- Pickup trip count  
- Dropoff trip count  
- Pickup/dropoff imbalance ratio  
- Average / median trip duration  
- Duration per mile  
- Late-night trip share (10 pm – 6 am)  
- Invalid trip rate (duration ≤ 0 or distance ≤ 0)  

---

## Repository Structure

- `data/` — raw and processed data  
- `notebooks/` — exploration and analysis  
- `src/` — reusable Python code  
- `dashboard/` — visualization app  
- `report/` — short memo and write-up  

---

## Project Status

**In progress.**  
Current work is focused on:
- Data extraction and cleaning  
- Taxi-zone aggregation  
- Exploratory analysis  
- Feature design for zone-level scoring  

---

## Limitations

- One month of data is not enough to capture seasonality  
- Taxi trips represent only part of urban travel demand  
- Zone-level metrics may be sensitive to outliers and rare extreme trips  
- Operational friction metrics are **screening tools**, not causal explanations  

---

## Next Steps

1. Refine taxi cleaning rules and outlier treatment  
2. Build daily zone-level aggregates  
3. Define a transparent friction score  
4. Rank priority zones for review  
5. Expand to multiple months  
6. Develop a dashboard and short memo  

---

## 1. Setup & Imports



In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

# Project paths
from nyc_mobility_friction.paths import get_project_paths
paths = get_project_paths()

taxi_dir = paths.raw / "taxi"

## 2. Load Data (January 2025 — Prototype Month)



In [9]:
# Discover all taxi files
taxi_files = sorted(taxi_dir.glob("yellow_tripdata_2025-*.parquet"))
print("Available taxi files:")
for f in taxi_files:
    print(f"   • {f.name}")

# Use January 2025 for the prototype (you can change this later)
taxi_path = [f for f in taxi_files if "2025-01" in f.name][0]
print(f"\nLoading prototype file: {taxi_path.name}")

taxi = pd.read_parquet(taxi_path)
print(f"Taxi shape (January 2025): {taxi.shape}")

Available taxi files:
   • yellow_tripdata_2025-01.parquet
   • yellow_tripdata_2025-02.parquet
   • yellow_tripdata_2025-03.parquet

Loading prototype file: yellow_tripdata_2025-01.parquet
Taxi shape (January 2025): (3475226, 20)


## 3. Data Audit



In [10]:
def audit_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_pct": (df.isna().mean() * 100).round(3),
        "n_unique": [df[col].nunique(dropna=True) if df[col].dtype != "object" 
                     else df[col].astype(str).nunique(dropna=True) 
                     for col in df.columns]
    }).sort_values(["missing_pct", "n_unique"], ascending=[False, False])
    print(f"\n{name.upper()} AUDIT")
    print("-" * (len(name) + 6))
    print(f"Shape: {df.shape}")
    return summary

taxi_audit = audit_df(taxi, "Taxi January 2025")
display(taxi_audit.head(15))


TAXI JANUARY 2025 AUDIT
-----------------------
Shape: (3475226, 20)


,column,dtype,missing_pct,n_unique
passenger_count,passenger_count,float64,15.54,10
RatecodeID,RatecodeID,float64,15.54,7
Airport_fee,Airport_fee,float64,15.54,7
congestion_surcharge,congestion_surcharge,float64,15.54,3
store_and_fwd_flag,store_and_fwd_flag,str,15.54,2
tpep_pickup_datetime,tpep_pickup_datetime,datetime64[us],0.00,1672077
tpep_dropoff_datetime,tpep_dropoff_datetime,datetime64[us],0.00,1671993
total_amount,total_amount,float64,0.00,21995
fare_amount,fare_amount,float64,0.00,11538
trip_distance,trip_distance,float64,0.00,4545


## 4. Time & Geography Check



In [11]:
# Normalize column names
taxi = taxi.rename(columns=str.lower).rename(columns=lambda x: x.replace(" ", "_"))

# Safe datetime parsing
for old, new in [("tpep_pickup_datetime", "pickup_datetime"),
                 ("tpep_dropoff_datetime", "dropoff_datetime")]:
    if old in taxi.columns:
        taxi[new] = pd.to_datetime(taxi[old], errors="coerce")

taxi["pickup_day"] = taxi["pickup_datetime"].dt.date
taxi["pickup_month"] = taxi["pickup_datetime"].dt.to_period("M").astype(str)

print("Month distribution:")
print(taxi["pickup_month"].value_counts().sort_index())
print(f"\nNumber of unique days: {taxi['pickup_day'].nunique()}")

Month distribution:
pickup_month
2024-12         21
2025-01    3475204
2025-02          1
Name: count, dtype: int64

Number of unique days: 33


## 5. Taxi Cleaning Rules & Friction Metrics



In [13]:
taxi["trip_duration_min"] = (
    (taxi["dropoff_datetime"] - taxi["pickup_datetime"]).dt.total_seconds() / 60
)

taxi["valid_trip"] = (
    (taxi["trip_duration_min"] > 0) &
    (taxi["trip_distance"] > 0) &
    (taxi["fare_amount"] > 0)
)

print(f"Valid trips: {taxi['valid_trip'].sum():,} / {len(taxi):,} "
      f"({taxi['valid_trip'].mean():.1%})")

numeric_cols = ["trip_distance", "fare_amount", "total_amount", 
                "tip_amount", "trip_duration_min"]
display(taxi[taxi["valid_trip"]][numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

Valid trips: 3,252,514 / 3,475,226 (93.6%)


,trip_distance,fare_amount,total_amount,tip_amount,trip_duration_min
count,"3,252,514.00","3,252,514.00","3,252,514.00","3,252,514.00","3,252,514.00"
mean,5.52,18.22,27.09,3.12,15.14
std,514.26,479.00,479.16,3.76,27.27
min,0.01,0.01,0.00,0.00,0.02
1%,0.25,4.40,9.15,0.00,2.00
5%,0.50,5.80,11.75,0.00,3.62
50%,1.70,12.80,20.35,2.59,11.70
95%,12.07,52.70,74.75,10.29,36.48
99%,19.58,70.90,102.92,17.19,59.33
max,"276,099.95","863,372.12","863,380.37",400.00,"5,626.32"


## 6. Key Takeaways (Feasibility Audit Complete)

- Data loading and basic audit successful  
- Taxi zones (PULocationID) are ready for aggregation  
- Cleaning rules are clearly defined  
- All metrics listed in the README are computable  
- Project is fully feasible for the prototype phase

